# Model Development

This notebook continues the analysis performed in `01_data_analysis_and_preprocessing.ipynb`.

The preprocessed dataset created in the first notebook is used here for train-validation splitting, final model-specific preprocessing, baseline model development, feature engineering and model evaluation.

## 4. Train-Validation Split

### 4.1 Feature and Target Separation

In [1]:
# Import libraries
import pandas as pd

# Import the data splitting function
from sklearn.model_selection import train_test_split

preprocessed_data = pd.read_csv('data/preprocessed_data.csv')

In [2]:
# Separate model features, target, and client identifiers
X = preprocessed_data.drop(columns = ['SK_ID_CURR', 'TARGET'])
y = preprocessed_data['TARGET']

client_ids = preprocessed_data['SK_ID_CURR']

print(X.shape, y.shape)

(307511, 126) (307511,)


At this stage, the dataset is separated into features (X) and the target (y) to prepare the data for baseline model development.

SK_ID_CURR is excluded from the model features because it is a technical client identifier and does not represent meaningful information for predicting repayment difficulties. However, the identifier is stored separately because it may later be useful for linking model predictions back to individual clients.

### 4.2 Train and Validation Split

In [3]:
# Split the data into training and validation sets
X_train, X_valid, y_train, y_valid = train_test_split(
    X, 
    y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
)

In [4]:
y_train.value_counts(normalize=True)

TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

In [5]:
y_valid.value_counts(normalize=True)


TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64

The data was split into training and validation sets using an 80/20 ratio. The validation set will be used to evaluate and compare models during development.

Since the target variable is imbalanced, stratified sampling was used to preserve approximately the same class distribution in both datasets. As a result, the proportion of clients with repayment difficulties remains around 8.07% in both the training and validation samples.

### 5. Final Preprocessing

The previous preprocessing stage focused on understanding and handling data quality issues. Special values and informative missingness were preserved through indicators, while categorical missing values were treated based on their meaning.

However, the data is still not ready for Logistic Regression. Numerical features still contain missing values, categorical features must be encoded, and numerical variables have different scales.

All preprocessing parameters will be learned only from the training data and then applied to the validation data to avoid data leakage.


#### 5.1 Numerical Missing Value Imputation

In [6]:
numeric_cols = X_train.select_dtypes(include='number').columns
numeric_cols

Index(['CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
       'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH',
       'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
       ...
       'AMT_REQ_CREDIT_BUREAU_DAY', 'AMT_REQ_CREDIT_BUREAU_WEEK',
       'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
       'AMT_REQ_CREDIT_BUREAU_YEAR', 'EXT_SOURCE_1_MISSING',
       'EXT_SOURCE_2_MISSING', 'EXT_SOURCE_3_MISSING',
       'CREDIT_BUREAU_REQ_MISSING', 'SOCIAL_CIRCLE_MISSING'],
      dtype='str', length=109)

In [7]:
missing_numeric = (
    X_train[numeric_cols]
    .isna()
    .mean()
    .sort_values(ascending = False)
)

missing_numeric[missing_numeric > 0]

COMMONAREA_AVG              0.698396
COMMONAREA_MODE             0.698396
COMMONAREA_MEDI             0.698396
NONLIVINGAPARTMENTS_MODE    0.693998
NONLIVINGAPARTMENTS_AVG     0.693998
                              ...   
EXT_SOURCE_2                0.002158
AMT_GOODS_PRICE             0.000898
AMT_ANNUITY                 0.000041
CNT_FAM_MEMBERS             0.000008
DAYS_LAST_PHONE_CHANGE      0.000004
Length: 62, dtype: float64

In [8]:
print('> 50%:', (missing_numeric > 0.50).sum())
print('20-50%:', ((missing_numeric > 0.20) & (missing_numeric <= 0.50)).sum())
print('5-20%:', ((missing_numeric > 0.05) & (missing_numeric <= 0.20)).sum())
print('< 5%:', ((missing_numeric > 0) & (missing_numeric <= 0.05)).sum())

> 50%: 38
20-50%: 7
5-20%: 8
< 5%: 9


In [9]:
missing_numeric[missing_numeric > 0.5]

COMMONAREA_AVG              0.698396
COMMONAREA_MODE             0.698396
COMMONAREA_MEDI             0.698396
NONLIVINGAPARTMENTS_MODE    0.693998
NONLIVINGAPARTMENTS_AVG     0.693998
NONLIVINGAPARTMENTS_MEDI    0.693998
LIVINGAPARTMENTS_MODE       0.683388
LIVINGAPARTMENTS_MEDI       0.683388
LIVINGAPARTMENTS_AVG        0.683388
FLOORSMIN_MEDI              0.678519
FLOORSMIN_AVG               0.678519
FLOORSMIN_MODE              0.678519
YEARS_BUILD_MEDI            0.664787
YEARS_BUILD_MODE            0.664787
YEARS_BUILD_AVG             0.664787
OWN_CAR_AGE                 0.660214
LANDAREA_AVG                0.593416
LANDAREA_MODE               0.593416
LANDAREA_MEDI               0.593416
BASEMENTAREA_MEDI           0.584652
BASEMENTAREA_MODE           0.584652
BASEMENTAREA_AVG            0.584652
EXT_SOURCE_1                0.563376
NONLIVINGAREA_AVG           0.551299
NONLIVINGAREA_MODE          0.551299
NONLIVINGAREA_MEDI          0.551299
ELEVATORS_AVG               0.532572
E

#### 5.1.1 OWN_CAR_AGE

Missing values in OWN_CAR_AGE have different meanings depending on car ownership.

For clients without a car (FLAG_OWN_CAR = N), the missing value is structural because car age is not applicable. These values are replaced with 0.

For clients who own a car (FLAG_OWN_CAR = Y) but have missing car age, the value is treated as unknown and replaced with the median car age calculated from the training data.

The same training median is applied to the validation set to avoid data leakage.

In [10]:
X_train['FLAG_OWN_CAR'].value_counts()

FLAG_OWN_CAR
N    162413
Y     83595
Name: count, dtype: int64

In [11]:
X_train[(X_train['FLAG_OWN_CAR']=='Y') & (X_train['OWN_CAR_AGE'].isna())]

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,EXT_SOURCE_1_MISSING,EXT_SOURCE_2_MISSING,EXT_SOURCE_3_MISSING,CREDIT_BUREAU_REQ_MISSING,SOCIAL_CIRCLE_MISSING,DAYS_EMPLOYED_SPECIAL
229867,Cash loans,F,Y,Y,1,225000.0,518562.0,25078.5,463500.0,Unknown,...,0.0,0.0,0.0,3.0,0,0,0,0,0,False
236868,Cash loans,F,Y,Y,0,225000.0,233833.5,26577.0,211500.0,Unknown,...,0.0,1.0,0.0,1.0,0,0,0,0,0,False
181231,Cash loans,F,Y,N,0,112500.0,301464.0,22068.0,238500.0,Unknown,...,0.0,1.0,0.0,5.0,1,0,1,0,0,False
30897,Cash loans,M,Y,N,1,495000.0,1006920.0,45630.0,900000.0,Unaccompanied,...,0.0,0.0,0.0,0.0,0,0,0,0,0,False
217549,Cash loans,M,Y,N,0,225000.0,900000.0,26446.5,900000.0,Unaccompanied,...,0.0,0.0,1.0,5.0,0,0,0,0,0,False


In [12]:
X_train[(X_train['FLAG_OWN_CAR']=='N') & (X_train['OWN_CAR_AGE'].isna())]

,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,EXT_SOURCE_1_MISSING,EXT_SOURCE_2_MISSING,EXT_SOURCE_3_MISSING,CREDIT_BUREAU_REQ_MISSING,SOCIAL_CIRCLE_MISSING,DAYS_EMPLOYED_SPECIAL
181648,Cash loans,F,N,N,2,90000.0,227520.0,13189.5,180000.0,Unaccompanied,...,0.0,0.0,1.0,1.0,0,0,0,0,0,False
122525,Cash loans,M,N,Y,0,135000.0,728847.0,26307.0,553500.0,"Spouse, partner",...,2.0,0.0,0.0,2.0,1,0,0,0,0,False
306311,Cash loans,M,N,N,0,135000.0,474183.0,34636.5,391500.0,Unaccompanied,...,0.0,0.0,0.0,4.0,1,0,0,0,0,False
300658,Cash loans,F,N,Y,0,180000.0,254700.0,27558.0,225000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,1,0,1,1,0,False
201033,Revolving loans,F,N,Y,0,74250.0,225000.0,11250.0,225000.0,Unaccompanied,...,0.0,3.0,1.0,2.0,1,0,0,0,0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170183,Cash loans,F,N,Y,1,157500.0,846517.5,33700.5,684000.0,Unaccompanied,...,0.0,0.0,1.0,4.0,1,0,0,0,0,False
31304,Revolving loans,F,N,Y,1,135000.0,405000.0,20250.0,405000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,1,0,1,1,0,False
121193,Cash loans,F,N,N,0,157500.0,272520.0,21528.0,225000.0,Unaccompanied,...,0.0,0.0,1.0,4.0,1,0,0,0,0,False
248504,Cash loans,F,N,N,0,90000.0,246357.0,24493.5,234000.0,Unaccompanied,...,0.0,0.0,0.0,1.0,1,0,0,0,0,True


In [13]:
# Use 0 for clients without a car.
# Use the training median for car owners with unknown car age.
# Apply the same training median to the validation set.

car_age_median = X_train.loc[
    X_train['FLAG_OWN_CAR'] == 'Y',
    'OWN_CAR_AGE'
].median()

train_car_missing = (
    (X_train['FLAG_OWN_CAR']=='Y') 
    & (X_train['OWN_CAR_AGE'].isna())
)

X_train.loc[train_car_missing, 'OWN_CAR_AGE'] = car_age_median

valid_car_missing = (
    (X_valid['FLAG_OWN_CAR']=='Y')
    & (X_valid['OWN_CAR_AGE'].isna())
)
X_valid.loc[valid_car_missing, 'OWN_CAR_AGE'] = car_age_median


train_no_car = (
    (X_train['FLAG_OWN_CAR']=='N')
    & (X_train['OWN_CAR_AGE'].isna())
)
valid_no_car = (
    (X_valid['FLAG_OWN_CAR']=='N')
    & (X_valid['OWN_CAR_AGE'].isna())
)

X_train.loc[train_no_car, 'OWN_CAR_AGE'] = 0

X_valid.loc[valid_no_car, 'OWN_CAR_AGE'] = 0

# Check
print('Train missing:', X_train['OWN_CAR_AGE'].isna().sum())
print('Valid missing:', X_valid['OWN_CAR_AGE'].isna().sum())

Train missing: 0
Valid missing: 0


#### 5.1.2 EXT_SOURCE Features

The EXT_SOURCE features contain missing values, and their missingness may carry useful information. Separate missing indicators were created during the previous preprocessing stage.

For the baseline Logistic Regression, missing values in each EXT_SOURCE feature are replaced with its median calculated from the training data. The same medians are applied to the validation set.

The missing indicators are retained to distinguish originally missing values from observed values.

In [14]:
# Calculate training medians and apply them to both datasets
ext1_median = X_train['EXT_SOURCE_1'].median()
ext2_median = X_train['EXT_SOURCE_2'].median()
ext3_median = X_train['EXT_SOURCE_3'].median()

X_train['EXT_SOURCE_1'] = X_train['EXT_SOURCE_1'].fillna(ext1_median)
X_valid['EXT_SOURCE_1'] = X_valid['EXT_SOURCE_1'].fillna(ext1_median)

X_train['EXT_SOURCE_2'] = X_train['EXT_SOURCE_2'].fillna(ext2_median)
X_valid['EXT_SOURCE_2'] = X_valid['EXT_SOURCE_2'].fillna(ext2_median)

X_train['EXT_SOURCE_3'] = X_train['EXT_SOURCE_3'].fillna(ext3_median)
X_valid['EXT_SOURCE_3'] = X_valid['EXT_SOURCE_3'].fillna(ext3_median)

# Check remaining missing values
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

print(X_train[ext_cols].isna().sum())
print(X_valid[ext_cols].isna().sum())


EXT_SOURCE_1    0
EXT_SOURCE_2    0
EXT_SOURCE_3    0
dtype: int64
EXT_SOURCE_1    0
EXT_SOURCE_2    0
EXT_SOURCE_3    0
dtype: int64


#### 5.1.3 Building-Related Features

In [15]:
building_cols = [
    'COMMONAREA_AVG',
    'COMMONAREA_MODE',
    'COMMONAREA_MEDI'
]

X_train[building_cols].corr()

,COMMONAREA_AVG,COMMONAREA_MODE,COMMONAREA_MEDI
COMMONAREA_AVG,1.000000,0.975988,0.995660
COMMONAREA_MODE,0.975988,1.000000,0.978934
COMMONAREA_MEDI,0.995660,0.978934,1.000000


In [16]:
X_train[building_cols].notna().all(axis=1).sum()

np.int64(74197)

In [17]:
# Examine missingness and the proportion of zeros among observed values
zero_summary = pd.DataFrame({
    'missing_rate': X_train[building_cols].isna().mean(),
    'zero_count': X_train[building_cols].eq(0).sum(),
    'zero_rate_among_observed': (
        X_train[building_cols].eq(0).sum()
        / X_train[building_cols].notna().sum()
    )
})

zero_summary

,missing_rate,zero_count,zero_rate_among_observed
COMMONAREA_AVG,0.698396,6757,0.091068
COMMONAREA_MODE,0.698396,7744,0.104371
COMMONAREA_MEDI,0.698396,6942,0.093562


The three COMMONAREA features show very strong linear relationships, with correlations ranging from 0.976 to 0.996. The correlation analysis was based on 74,197 complete training observations.

Zeros account for approximately 9–10% of the observed values. Their presence alone is not considered a reason to remove the features.

Although the strong correlations suggest possible redundancy and multicollinearity, correlation alone does not prove that the features have no additional predictive value. Therefore, all building-related features are retained for the initial baseline model.

A reduced feature set may be evaluated later to determine whether removing highly correlated features improves model simplicity without materially reducing predictive performance.

In [18]:
building_numeric_cols = X_train.select_dtypes(
    include='number'
).columns[
    X_train.select_dtypes(include='number')
    .columns.str.endswith(('_AVG', '_MODE', '_MEDI'))
].tolist()

print('Number of building features:', len(building_numeric_cols))
print(building_numeric_cols)

Number of building features: 43
['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'TOTALAREA_MODE']


For the baseline model, all 43 numerical building-related features are retained.

A BUILDING_MISSING_COUNT feature is created before imputation to preserve information about the number of missing building characteristics for each client. This transformation is based only on the client's own data and does not use the target.

Missing values are then replaced with the median of each feature calculated from the training data. The same medians are applied to the validation set.

In [19]:
X_train['BUILDING_MISSING_COUNT'] = (
    X_train[building_numeric_cols].isna().sum(axis=1)
)

X_valid['BUILDING_MISSING_COUNT'] = (
    X_valid[building_numeric_cols].isna().sum(axis=1)
)

/var/folders/yr/jpmjpd_j7s59ts2plwx42_vw0000gn/T/ipykernel_3886/2898478897.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train['BUILDING_MISSING_COUNT'] = (
/var/folders/yr/jpmjpd_j7s59ts2plwx42_vw0000gn/T/ipykernel_3886/2898478897.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_valid['BUILDING_MISSING_COUNT'] = (


In [20]:
building_medians = X_train[building_numeric_cols].median()

X_train[building_numeric_cols] = (
    X_train[building_numeric_cols].fillna(building_medians)
)

X_valid[building_numeric_cols] = (
    X_valid[building_numeric_cols].fillna(building_medians)
)

In [21]:
print(X_train[building_numeric_cols].isna().sum().sum())
print(X_valid[building_numeric_cols].isna().sum().sum())

0
0


In [22]:
remaining_missing = (
    X_train[numeric_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

remaining_missing = remaining_missing[remaining_missing > 0]

remaining_missing

DAYS_EMPLOYED                 44143
AMT_REQ_CREDIT_BUREAU_QRT     33244
AMT_REQ_CREDIT_BUREAU_YEAR    33244
AMT_REQ_CREDIT_BUREAU_HOUR    33244
AMT_REQ_CREDIT_BUREAU_DAY     33244
AMT_REQ_CREDIT_BUREAU_WEEK    33244
AMT_REQ_CREDIT_BUREAU_MON     33244
DEF_30_CNT_SOCIAL_CIRCLE        811
OBS_60_CNT_SOCIAL_CIRCLE        811
DEF_60_CNT_SOCIAL_CIRCLE        811
OBS_30_CNT_SOCIAL_CIRCLE        811
AMT_GOODS_PRICE                 221
AMT_ANNUITY                      10
CNT_FAM_MEMBERS                   2
DAYS_LAST_PHONE_CHANGE            1
dtype: int64

#### 5.1.4 DAYS_EMPLOYED

The special value '365243' was previously replaced with NaN, while its original presence was preserved in DAYS_EMPLOYED_SPECIAL.

For the baseline model, the remaining missing values are replaced with the median of observed employment days calculated from the training data. The same median is applied to the validation set.

This is a technical imputation and does not represent the actual employment history of clients with the original special value.

In [23]:
employment_median = X_train['DAYS_EMPLOYED'].median()

X_train['DAYS_EMPLOYED'] = X_train['DAYS_EMPLOYED'].fillna(employment_median)
X_valid['DAYS_EMPLOYED'] = X_valid['DAYS_EMPLOYED'].fillna(employment_median)

In [24]:
print('Train median:', employment_median)
print('Train missing:', X_train['DAYS_EMPLOYED'].isna().sum())
print('Valid missing:', X_valid['DAYS_EMPLOYED'].isna().sum())

Train median: -1648.0
Train missing: 0
Valid missing: 0


#### 5.1.5 Credit Bureau Request Features

The six credit bureau request features have a shared missingness pattern. Their original missingness was preserved in CREDIT_BUREAU_REQ_MISSING.

Missing values are replaced with the median of each feature calculated from the training data. The same medians are applied to the validation set.

The missing indicator is retained because an unknown number of requests is not necessarily equivalent to zero requests.

In [25]:
req_cols = [
    'AMT_REQ_CREDIT_BUREAU_HOUR',
    'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK',
    'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR'
]

req_medians = X_train[req_cols].median()

X_train[req_cols] = X_train[req_cols].fillna(req_medians)
X_valid[req_cols] = X_valid[req_cols].fillna(req_medians)

print('Train missing:', X_train[req_cols].isna().sum().sum())
print('Valid missing:', X_valid[req_cols].isna().sum().sum())


Train missing: 0
Valid missing: 0


#### 5.1.6 Social Circle Features

The four social circle features have a shared missingness pattern, which was preserved in SOCIAL_CIRCLE_MISSING.

For the baseline model, missing values are replaced with the median of each feature calculated from the training data. The same medians are applied to the validation set.

All four training medians are equal to zero. These values are used because they are the observed medians, not because missing information is assumed to represent zero cases.

In [26]:
social_cols = [
    'OBS_30_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE',
    'OBS_60_CNT_SOCIAL_CIRCLE',
    'DEF_60_CNT_SOCIAL_CIRCLE'
]

social_medians = X_train[social_cols].median()

social_medians

OBS_30_CNT_SOCIAL_CIRCLE    0.0
DEF_30_CNT_SOCIAL_CIRCLE    0.0
OBS_60_CNT_SOCIAL_CIRCLE    0.0
DEF_60_CNT_SOCIAL_CIRCLE    0.0
dtype: float64

In [27]:
X_train[social_cols] = X_train[social_cols].fillna(social_medians)
X_valid[social_cols] = X_valid[social_cols].fillna(social_medians)

In [28]:
print('Train missing:', X_train[social_cols].isna().sum().sum())
print('Valid missing:', X_valid[social_cols].isna().sum().sum())

Train missing: 0
Valid missing: 0


#### 5.1.7 Remaining Numerical Missing Values

The remaining numerical features contain only a small number of missing values. No additional structural missingness was identified for these features during the previous analysis.

For the baseline model, missing values are replaced with the median of each feature calculated from the training data. The same medians are applied to the validation set.

In [29]:
remaining_cols = [
    'AMT_GOODS_PRICE',
    'AMT_ANNUITY',
    'CNT_FAM_MEMBERS',
    'DAYS_LAST_PHONE_CHANGE'
]

remaining_medians = X_train[remaining_cols].median()

remaining_medians

AMT_GOODS_PRICE           450000.0
AMT_ANNUITY                24903.0
CNT_FAM_MEMBERS                2.0
DAYS_LAST_PHONE_CHANGE      -757.0
dtype: float64

In [30]:
X_train[remaining_cols] = X_train[remaining_cols].fillna(remaining_medians)
X_valid[remaining_cols] = X_valid[remaining_cols].fillna(remaining_medians)

remaining_cols = X_train.select_dtypes(include='number').columns

train_missing = X_train[numeric_cols].isna().sum()
valid_missing = X_valid[numeric_cols].isna().sum()

print('Train missing:', train_missing.sum())
print('Valid missing:', valid_missing.sum())

Train missing: 0
Valid missing: 0
